# 🧾 Notebook 2: Event Sourcing + CQRS

Instead of storing *current state*, we store the **stream of events** that led to it.
The query side is built by **replaying** those events into read models.

Power:
- Full audit log for free.
- You can **add new projections later** by replaying events.

## 🛠️ Setup

```bash
cd 05-microservices/cqrs
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
events = []

def emit(e):
    events.append(e)
    for p in projections:
        p(e)

# --- Projections (built by replay) ---
user_summary = {}
top_spenders = {}

def p_summary(e):
    if e['type'] == 'UserCreated':
        user_summary[e['uid']] = {'name': e['name'], 'orders':0, 'spent':0.0}
    elif e['type'] == 'OrderPlaced':
        u = user_summary[e['uid']]
        u['orders'] += 1; u['spent'] += e['total']

def p_top(e):
    if e['type'] == 'OrderPlaced':
        top_spenders[e['uid']] = top_spenders.get(e['uid'], 0) + e['total']

projections = [p_summary, p_top]

# --- Commands -> events ---
emit({'type':'UserCreated','uid':1,'name':'Ada'})
emit({'type':'UserCreated','uid':2,'name':'Grace'})
emit({'type':'OrderPlaced','uid':1,'total':42})
emit({'type':'OrderPlaced','uid':2,'total':99})
emit({'type':'OrderPlaced','uid':1,'total':8})

print('summary:', user_summary)
print('top:', sorted(top_spenders.items(), key=lambda x:-x[1]))
print('event log length:', len(events))


### Replay to add a new projection later

In [ ]:
# Business now wants average order size. Define a projection and replay history.
avg = {}
def p_avg(e):
    if e['type']=='OrderPlaced':
        a = avg.setdefault(e['uid'], {'n':0,'sum':0})
        a['n']+=1; a['sum']+=e['total']

for e in events:
    p_avg(e)

print({u: round(a['sum']/a['n'],2) for u,a in avg.items()})


### Trade-offs
- ✅ Audit, time-travel, new views from history.
- ❌ More moving parts, eventual consistency between write and read, schema evolution of events is subtle.